In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [13]:
# benchmark dataset
# 目标: 12个tag进行rag，3秒内
import random

from entropy.domain.services.tag_checker import TagChecker


input_data = """
shimmering, pleading, soaked, slightly open mouth, thin straps, sheer dress, blushing, shoulderless dress,
seifuku, japanese school uniform, bubbles, knitwear,
low angle, daydreaming, wild life, surrounded by bubbles, sunbeams, cross-legged, flowing dress,
adjusting glove, rim light, tyndall effect, floating flower petals, misty,
magical atmosphere, motes, thoughtful, dangling legs, cinematic lighting, purple and gold theme,
reading book, white clouds, large bubbles, scenic
"""

input_data = TagChecker.extract_all_tags(input_data)
input_data = list(set(input_data))
input_data = sorted(input_data)

input_data = random.Random(2).sample(input_data, k=12)

assert len(input_data) == 12

print(",".join(input_data))

cinematic lighting,dangling legs,white clouds,large bubbles,soaked,tyndall effect,shimmering,slightly open mouth,seifuku,japanese school uniform,flowing dress,rim light


In [14]:
from entropy.domain.services.rag_service import RagService

In [15]:
# 非batch

result1 = []

for invalid_tag in input_data:
    tags, scores = RagService.do_rag(query_text=invalid_tag, rerank_output=10)
    result1.append(tags)

Compute Scores: 100%|██████████| 4/4 [00:00<00:00,  6.73it/s]


In [16]:
# batch
batch_rag_output = RagService.batch_rag(query_text_list=input_data,recall_count=200, rerank_output=10)

result2 = [tags for tags, scores in batch_rag_output]

Compute Scores: 100%|██████████| 19/19 [00:02<00:00,  9.46it/s]


In [17]:
import json


for (invalid_tag, r1, r2) in zip(input_data, result1, result2):
    if not json.dumps(r1) == json.dumps(r2):
        print(f"invalid danbooru tag: {invalid_tag}")
        print("rag result 1:", r1)
        print("rag result 2:", r2)

invalid danbooru tag: cinematic lighting
rag result 1: ['screen light', 'studio lights', 'lights', 'rotating light', 'illumination', 'light', 'luminous', 'stage lights', 'glowing', 'aiming at viewer']
rag result 2: ['screen light', 'studio lights', 'lights', 'rotating light', 'illumination', 'light', 'luminous', 'stage lights', 'glowing', 'light beam']
invalid danbooru tag: white clouds
rag result 1: ['white sky', 'cloudy sky', 'cloud', 'white theme', 'cloud background', 'white background', 'overcast', 'on cloud', 'whorled clouds', 'white coat']
rag result 2: ['white sky', 'cloudy sky', 'cloud', 'white theme', 'cloud background', 'white background', 'on cloud', 'whorled clouds', 'white coat', 'white scales']
invalid danbooru tag: large bubbles
rag result 1: ['large bulge', 'bobbles', 'big belly', 'in bubble', 'grand sphere', 'bulge', 'oversized object', 'huge nipples', 'bubble', 'bulges touching']
rag result 2: ['large bulge', 'bobbles', 'big belly', 'in bubble', 'grand sphere', 'overs

我试图从方法1改为方法2，请帮我看看效果是否有明显下降。从2个角度：
1. 这个是检测到无效tag，对用户输出guess you like tags的
2. 除了将无效变为有效之外，tag也能提供灵感


In [ ]:
#